# IALS (Implicit Alternating Least Squares )

В этом ноутбуке я построю IALS модель, протестирую её и сохраню 

## Импорт библиотек 

In [1]:
import pandas as pd
import numpy as np
import scipy 
import os
import implicit 
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from implicit.evaluation import train_test_split as implicit_split 
from implicit.evaluation import precision_at_k, mean_average_precision_at_k, ranking_metrics_at_k
import implicit.gpu
import optuna
import optuna.visualization as vis
import plotly
import time


In [2]:
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

## Загрузка данных

#### Выгружаем матрицу

In [3]:
results_path = Path("../../../results/matrices")
user_item_sparse = scipy.sparse.load_npz(results_path/'artnail_user_item_sparse.npz')

#### Выгружаем очищенные таблицы

In [4]:
clean_path = Path("../../../Tables/CleanTable")
users_clean = pd.read_csv(clean_path / 'users_clean_final.csv')
items_clean = pd.read_csv(clean_path / 'items_clean_final.csv')

In [5]:
f'Размеры матрицы: {user_item_sparse.shape}'

'Размеры матрицы: (2893, 258)'

## IALS model 

#### Train/Validation split для метрик

In [6]:
train_sparse, val_sparse = implicit_split(user_item_sparse, train_percentage=0.9, random_state=42)

In [7]:
print(f"  Train: {train_sparse.nnz} ({train_sparse.nnz/user_item_sparse.nnz:.1%})")
print(f"  Val:   {val_sparse.nnz} ({val_sparse.nnz/user_item_sparse.nnz:.1%})")

  Train: 877 (90.4%)
  Val:   93 (9.6%)


#### Построение базовой модели 

In [8]:
IALS_model = implicit.als.AlternatingLeastSquares(
    factors=64,           # Размер эмбеддингов (пользователи × услуги)
    regularization=0.1,   # L2 регуляризация (0.01-0.5)
    iterations=50,        # ALS итераций (сходимость)
    random_state=42,      # Репродуцируемость
    use_gpu=False,        # CPU 
    num_threads=4         # Потоки (4-8 оптимально)
)

c:\obuchenie\ArtNail\.venv\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


Обучение 

In [9]:
CONFIDENCE_SCALE = 15
IALS_model.fit(train_sparse*CONFIDENCE_SCALE)

print(f"  User embeddings: {IALS_model.user_factors.shape}")
print(f"  Item embeddings: {IALS_model.item_factors.shape}")

  0%|          | 0/50 [00:00<?, ?it/s]

  User embeddings: (2893, 64)
  Item embeddings: (258, 64)


#### Оценка качества базовой модели 

IALS будет выполнять функции подбора кандидатов, а функцию ранжирования будет выполнять `CutBoost` модель, поэтому в оценке `IALS` главной метрикой буду считать Recall@K, где K будет большим.

Цель не отобрать конкретные 10 лучших вариантов, а отобрать 50 кандидатов, главное чтобы среди них были хорошие кандидаты и CutBoost увидел их

In [10]:
metrics = ranking_metrics_at_k(IALS_model, train_sparse, val_sparse, K=50)
metrics

  0%|          | 0/91 [00:00<?, ?it/s]

{'precision': 0.5806451612903226,
 'map': 0.029722000906388105,
 'ndcg': 0.12825269952781929,
 'auc': 0.6866756822032526}

Пояснение: В implicit версии 0.7.2 метрика Precision@K считает по факту как Recall@K, поэтому мы будем интарпретировать precision как recall. 

In [11]:
recall_50 = metrics['precision']
f'Recall@50 = {recall_50:.6f}'

'Recall@50 = 0.580645'

__Из 50 кандидатов 58% релевантны__. Это хороший показатель для CutBoost. 

Также  `AUC` = 0.6819 показывает, что модель уверенно отличает положительные и негативные оценки.

Из остальных метрик можно понять, что модель плохо ранжирует отобранных кандидатов, но для нашей ситуации это не страшно, т. к. ранжировать кандидатов будет CutBoost.

#### Random Search с  помощью Optuna

In [12]:

def objective(trial):
    """Optuna objective: максимизируем Recall@50"""
    
    # Предлагаем параметры
    params = {
    # Размер эмбеддингов: от компактных до довольно крупных
    # (чем больше factors, тем качественнее, но медленнее и больше памяти)
    'factors': trial.suggest_int('factors', 32, 160, step=16),  
    # 32, 48, 64, 80, 96, 112, 128, 144, 160

    # Регуляризация: от очень слабой до довольно сильной
    # (я бы уже использовал log-шкалу — так лучше исследуются края)
    'regularization': trial.suggest_float('regularization', 0.01, 0.5),

    # Число итераций: чуть меньше базового до заметно больше
    # (если данные не очень большие, можно позволить до 80)
    'iterations': trial.suggest_int('iterations', 30, 80),

    # Дополнительные параметры ALS:
    # weight для нормализации регуляризации по числу факторов
    'dtype': np.float32,  # чтобы каждый trial был одинаков по типу
    'use_gpu': False,
    'num_threads': 4,
    'random_state': 42,
    'calculate_training_loss': False
}

    # Модель
    model = implicit.als.AlternatingLeastSquares(**params)
    
    # Обучение 
    CONFIDENCE_SCALE = 15
    model.fit(train_sparse*CONFIDENCE_SCALE, show_progress=False)

    # Метрики
    metrics = ranking_metrics_at_k(
        model, 
        train_user_items=train_sparse, 
        test_user_items=val_sparse, 
        K=50,
        show_progress=False
    )

    recall50 = metrics['precision']

     
    # Callback для pruning (Optuna может прервать плохой trial)
    trial.report(recall50, step=1)
    if trial.should_prune():
        raise optuna.TrialPruned()
    
    
    print(f"✅ Trial {trial.number}: Recall@50={recall50:.4f} | {params}")
    return recall50


In [13]:

study = optuna.create_study(
    study_name='IALS Optuna Optimization',
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(),
    storage='sqlite:///ex.db',
    load_if_exists=True
)

[I 2026-03-10 19:47:22,201] Using an existing study with name 'IALS Optuna Optimization' instead of creating a new one.


In [14]:
study.optimize(objective, n_trials=50)
print(f"Лучший Recall@50: {study.best_value}")


[I 2026-03-10 19:47:23,601] Trial 82 pruned. 
[I 2026-03-10 19:47:25,197] Trial 83 finished with value: 0.6559139784946236 and parameters: {'factors': 96, 'regularization': 0.4574088016448851, 'iterations': 36}. Best is trial 44 with value: 0.6559139784946236.


✅ Trial 83: Recall@50=0.6559 | {'factors': 96, 'regularization': 0.4574088016448851, 'iterations': 36, 'dtype': <class 'numpy.float32'>, 'use_gpu': False, 'num_threads': 4, 'random_state': 42, 'calculate_training_loss': False}


[I 2026-03-10 19:47:26,861] Trial 84 finished with value: 0.6559139784946236 and parameters: {'factors': 96, 'regularization': 0.48595909492273803, 'iterations': 36}. Best is trial 44 with value: 0.6559139784946236.


✅ Trial 84: Recall@50=0.6559 | {'factors': 96, 'regularization': 0.48595909492273803, 'iterations': 36, 'dtype': <class 'numpy.float32'>, 'use_gpu': False, 'num_threads': 4, 'random_state': 42, 'calculate_training_loss': False}


[I 2026-03-10 19:47:28,448] Trial 85 finished with value: 0.6559139784946236 and parameters: {'factors': 96, 'regularization': 0.48246045820043637, 'iterations': 35}. Best is trial 44 with value: 0.6559139784946236.


...

✅ Trial 127: Recall@50=0.6452 | {'factors': 96, 'regularization': 0.4576216617373977, 'iterations': 42, 'dtype': <class 'numpy.float32'>, 'use_gpu': False, 'num_threads': 4, 'random_state': 42, 'calculate_training_loss': False}


[I 2026-03-10 19:49:12,939] Trial 128 finished with value: 0.6559139784946236 and parameters: {'factors': 96, 'regularization': 0.4989281505891911, 'iterations': 37}. Best is trial 107 with value: 0.6666666666666666.


✅ Trial 128: Recall@50=0.6559 | {'factors': 96, 'regularization': 0.4989281505891911, 'iterations': 37, 'dtype': <class 'numpy.float32'>, 'use_gpu': False, 'num_threads': 4, 'random_state': 42, 'calculate_training_loss': False}


[I 2026-03-10 19:49:15,377] Trial 129 pruned. 
[I 2026-03-10 19:49:18,250] Trial 130 pruned. 
[I 2026-03-10 19:49:21,428] Trial 131 finished with value: 0.6451612903225806 and parameters: {'factors': 96, 'regularization': 0.4810619688058947, 'iterations': 44}. Best is trial 107 with value: 0.6666666666666666.


✅ Trial 131: Recall@50=0.6452 | {'factors': 96, 'regularization': 0.4810619688058947, 'iterations': 44, 'dtype': <class 'numpy.float32'>, 'use_gpu': False, 'num_threads': 4, 'random_state': 42, 'calculate_training_loss': False}
Лучший Recall@50: 0.6666666666666666


In [15]:
best_params = study.best_params
f'best_params = {best_params}'

"best_params = {'factors': 96, 'regularization': 0.468162720686304, 'iterations': 79}"

#### Получение оптимальной модели

In [ ]:
best_IALS_model = implicit.als.AlternatingLeastSquares(**best_params, 
                                                    dtype = np.float32,
                                                    use_gpu = False,
                                                    num_threads = 4,
                                                    random_state = 42,
                                                    calculate_training_loss= False
                                                    )

In [27]:
CONFIDENCE_SCALE = 15
best_IALS_model.fit(train_sparse*CONFIDENCE_SCALE)

  0%|          | 0/79 [00:00<?, ?it/s]

Новые метрики

In [29]:
best_metrics = ranking_metrics_at_k(best_IALS_model, train_sparse, val_sparse, K=50)
best_metrics

  0%|          | 0/91 [00:00<?, ?it/s]

{'precision': 0.6666666666666666,
 'map': 0.06888421434054022,
 'ndcg': 0.1873715190192611,
 'auc': 0.7577863185877827}

Старые матрики

In [30]:
metrics

{'precision': 0.5806451612903226,
 'map': 0.029722000906388105,
 'ndcg': 0.12825269952781929,
 'auc': 0.6866756822032526}

In [33]:
pd.DataFrame({
    'Metric':['recall', 'map', 'nscg', 'auc'],
    'DefaultModel': [metrics['precision'], metrics['map'], metrics['ndcg'], metrics['auc']],
    'BestModel': [best_metrics['precision'], best_metrics['map'], best_metrics['ndcg'], best_metrics['auc']]
})

,Metric,DefaultModel,BestModel
0,recall,0.580645,0.666667
1,map,0.029722,0.068884
2,nscg,0.128253,0.187372
3,auc,0.686676,0.757786


Видим: 
1) __Вырос recall@50__ ( он же precision в этйо версии) с 0,58 до 0,67
2) Остальные параметры тоже высорли (хоть модель всё ещё плохо ранжирует кандидатов, нам это не критично)
3) __В среднем прирост около 15%__

Что это значит прикладно:

__Теперь из 50 кандидатов 67% реливантны.__

#### Проверка на переобучение 

In [34]:
train_raw = train_sparse.T.tocsr().astype('float32')
val_raw = val_sparse.T.tocsr().astype('float32')

train_metrics = ranking_metrics_at_k(best_IALS_model, train_raw, train_raw, K=50)
print(f"Train Recall@50:  {train_metrics['precision']:.4f}")
print(f"Val   Recall@50:  {best_metrics['precision']:.4f}")
print(f"GAP:              {train_metrics['precision'] - metrics['precision']:.4f}")

  0%|          | 0/82 [00:00<?, ?it/s]

IndexError: index 2224 is out of bounds for axis 1 with size 258